In [1]:
import lzma
import pickle
import re
import unicodedata

import pandas as pd

In [2]:
with lzma.open("../../data/cleaned/starling_cleaned.pkl.xz", "rb") as f:
    starling = pickle.load(f)

with lzma.open("../../data/cleaned/unihan_cleaned.pkl.xz", "rb") as f:
    unihan = pickle.load(f)

In [3]:
def take_non_missing_value(row):
    return "; ".join({unicodedata.normalize("NFC", str(value)) for value in row if pd.notna(value)})

merged = unihan.merge(starling, on="Character", how="outer", suffixes=("_unihan", "_starling"))

merged["definition"]      = merged[["kDefinition", "English meaning"]].apply(take_non_missing_value, axis=1)
merged["pinyin"]          = merged[["kMandarin", "Modern (Beijing) reading"]].apply(take_non_missing_value, axis=1)
merged["decomposed_pinyin"] = merged[["decomposed_pinyin_unihan", "decomposed_pinyin_starling"]].apply(take_non_missing_value, axis=1)
merged["toneless_pinyin"] = merged.decomposed_pinyin.apply(lambda x: "".join(re.findall(r'[a-z]+', x)))

merged = merged.rename(columns={
    "Character":   "hanzi",
    "kHangul":     "hangul",
    "kKorean":     "anglo_hangul",
    "kJapanese":   "katakana",
    "kJapaneseOn": "anglo_katakana",
})[["hanzi", "definition", "pinyin", "decomposed_pinyin", "toneless_pinyin", "hangul", "anglo_hangul", "katakana", "anglo_katakana"]]

merged

,hanzi,definition,pinyin,decomposed_pinyin,toneless_pinyin,hangul,anglo_hangul,katakana,anglo_katakana
0,Ф,"to send, cause (?)",bēng,"('b', 'eng', 1)",beng,NaN,NaN,NaN,NaN
1,Щ,"be robust, strong",bì,"('b', 'i', 4)",bi,NaN,NaN,NaN,NaN
2,к,"flask, bottle (for wine)",yǒu,"('y', 'ou', 3)",you,NaN,NaN,NaN,NaN
3,へ,"be grieved, sad",daō,"('d', 'ao', 1)",dao,NaN,NaN,NaN,NaN
4,一,"be one, single, whole; one; a, an; alone",yī,"('y', 'i', 1)",yi,일,IL,イチ,ICHI
...,...,...,...,...,...,...,...,...,...
8027,龜,"turtle, tortoise; bone oracle in general; turt...",guī,"('g', 'ui', 1)",gui,구,KWU,キ,KI
8028,龝,"autumn, fall; year",qiū,"('q', 'iu', 1)",qiu,추,CHWU,シュウ,SHUU
8029,龠,"flute; pipe, ancient measure; Kangxi radical 214",yuè,"('y', 'ue', 4)",yue,약,YAK,ヤク,YAKU
8030,龢,"in harmony; calm, peaceful",hé,"('h', 'e', 2)",he,화,HWA,カ,KA


In [4]:
merged.to_pickle("../../data/cleaned/merged.pkl.xz")